# Session 08 – Data Cleaning (Part 3: Duplicates, Typos, Consistency)

This notebook continues from Session 07's 400-row bike listings dataset.

**Tasks:** detect exact and ID-level duplicates, remove duplicates in the required order, repair inconsistent brand/city labels, and standardize insurance values.


In [ ]:
import numpy as np
import pandas as pd
from difflib import get_close_matches

# Load the final dataset created in Session 07
bikes = pd.read_csv("../Session-07/bike_listings_s7.csv")
print("Original shape:", bikes.shape)
print("Unique listing IDs:", bikes["listing_id"].nunique())
bikes.head()


## Task 1 – Add and detect duplicates
Add 25 exact copies and 10 relisted rows. Relisted rows keep the same `listing_id` but have a 5% lower `price_inr`.

In [ ]:
exact_copies = bikes.sample(25, random_state=42).copy()
relisted = bikes.sample(10, random_state=7).copy()
relisted["price_inr"] = (relisted["price_inr"] * 0.95).round().astype(int)

combined = pd.concat([bikes, exact_copies, relisted], ignore_index=True)

print("Combined shape:", combined.shape)
print("Exact duplicate rows:", combined.duplicated().sum())
print("Rows sharing a listing_id:")
print(combined[combined.duplicated(subset=["listing_id"], keep=False)]
      .sort_values("listing_id")[["listing_id", "price_inr"]])


## Task 2 – Remove duplicates in the required order
First remove exact copies. Then keep the last row for each `listing_id`, because relisted rows were appended last.

In [ ]:
bikes_nodup = combined.drop_duplicates()
print("After drop_duplicates():", bikes_nodup.shape)

bikes_unique = bikes_nodup.drop_duplicates(
    subset=["listing_id"], keep="last"
)
print("After keeping latest listing:", bikes_unique.shape)
print("Unique listing IDs:", bikes_unique["listing_id"].nunique())

assert len(bikes_nodup) == 410
assert len(bikes_unique) == 400
assert bikes_unique["listing_id"].is_unique


## Task 3 – Introduce and repair category inconsistencies
Create lowercase values, trailing spaces, and the misspellings `Velose` and `Ahmdabad`. Use `value_counts()` before and after repair.

In [ ]:
messy = bikes_unique.copy()
rng = np.random.default_rng(7)

brand_lower_idx = rng.choice(messy.index, 30, replace=False)
city_lower_idx = rng.choice(messy.index, 30, replace=False)
brand_space_idx = rng.choice(messy.index, 20, replace=False)
city_space_idx = rng.choice(messy.index, 20, replace=False)

veloce_idx = messy.index[messy["brand"].str.strip().str.title().eq("Veloce")].to_numpy()
ahmedabad_idx = messy.index[messy["city"].str.strip().str.title().eq("Ahmedabad")].to_numpy()
brand_typo_idx = rng.choice(veloce_idx, 12, replace=False)
city_typo_idx = rng.choice(ahmedabad_idx, 12, replace=False)

messy.loc[brand_lower_idx, "brand"] = messy.loc[brand_lower_idx, "brand"].str.lower()
messy.loc[city_lower_idx, "city"] = messy.loc[city_lower_idx, "city"].str.lower()
messy.loc[brand_space_idx, "brand"] = messy.loc[brand_space_idx, "brand"] + "   "
messy.loc[city_space_idx, "city"] = messy.loc[city_space_idx, "city"] + "   "
messy.loc[brand_typo_idx, "brand"] = "Velose"
messy.loc[city_typo_idx, "city"] = "Ahmdabad"

print("Brand counts before fixing:")
print(messy["brand"].value_counts())
print("\\nCity counts before fixing:")
print(messy["city"].value_counts())

canonical_brands = bikes["brand"].dropna().unique().tolist()
canonical_cities = bikes["city"].dropna().unique().tolist()

def repair_category(value, canonical_values):
    cleaned = str(value).strip().title()
    exact_map = {item.lower(): item for item in canonical_values}
    if cleaned.lower() in exact_map:
        return exact_map[cleaned.lower()]
    close = get_close_matches(cleaned, canonical_values, n=1, cutoff=0.8)
    return close[0] if close else cleaned

messy["brand"] = messy["brand"].map(
    lambda value: repair_category(value, canonical_brands)
)
messy["city"] = messy["city"].map(
    lambda value: repair_category(value, canonical_cities)
)

print("\\nBrand counts after fixing:")
print(messy["brand"].value_counts())
print("\\nCity counts after fixing:")
print(messy["city"].value_counts())

assert messy["brand"].nunique() == 4
assert messy["city"].nunique() == 4


## Task 4 – Standardize insurance values and save the final CSV

In [ ]:
insurance_values = np.array(["Yes", "yes", "Y", "NO", "no", "N", " yes "])
messy["is_insured"] = rng.choice(
    insurance_values, size=len(messy), replace=True
)

insurance_map = {"yes": 1, "y": 1, "no": 0, "n": 0}
messy["is_insured"] = (
    messy["is_insured"].astype(str).str.strip().str.lower()
    .map(insurance_map).astype("int8")
)

print("Insurance values after standardization:")
print(messy["is_insured"].value_counts().sort_index())

assert len(messy) == 400
assert messy["listing_id"].nunique() == 400
assert messy["brand"].nunique() == 4
assert messy["city"].nunique() == 4
assert set(messy["is_insured"].unique()).issubset({0, 1})

messy.to_csv("bike_listings_clean.csv", index=False)
print("Saved bike_listings_clean.csv")


## Final verification
- 435 rows after adding 25 exact copies and 10 relisted rows.
- 410 rows after removing exact duplicates.
- 400 rows after keeping the latest row per `listing_id`.
- Final data has 400 unique listing IDs, exactly 4 brands, exactly 4 cities, and only 0/1 in `is_insured`.
- Final output: `bike_listings_clean.csv`.
